# Build Evaluation Sample

Stratified random sample of RC×CA match pairs for **manual labeling**.
The sample is **blinded** — which method flagged each pair is stored in a
separate key file that you should not open until labeling is done.

**Three source cells:**

| Cell | n | Stratification |
|---|---|---|
| **Agreement** (fuzzy ∩ cluster) | 50 | none — restricted to pairs where **preprocessed names differ** |
| **Fuzzy-only** (fuzzy − cluster) | 100 | 50 from `fuzzy_score ∈ [82, 90)`, 50 from `[90, 100]` |
| **Cluster-only** (cluster − fuzzy) | 100 | 50 from `cluster_size ≤ 5`, 50 from `cluster_size > 5` |

### Why the agreement filter

Both methods operate on **preprocessed** names. If preprocessed names are identical,
both methods will trivially link the pair — there's nothing to evaluate. About 80% of
agreement pairs fall in this trivial bucket, so a random sample of 50 would be
dominated by them.

Restricting the agreement sample to pairs where preprocessed names *differ* tests the
non-trivial agreement cases: fuzzy linked them via token similarity ≥ 82, **and**
clustering linked them via the LLM placing both names in the same cluster.

The fuzzy-only and cluster-only cells are **structurally guaranteed** to have
non-identical preprocessed names (if they were identical, both methods would link
them and the pair would be in the agreement cell), so no filter is needed there.

**Outputs:**
- `evaluation_sample.csv` — for the labeler (250 rows, shuffled, no source_cell)
- `evaluation_sample_key.csv` — unblinding key (pair_id → source_cell, stratum, scores)


In [1]:
import pandas as pd
import numpy as np

SEED = 42

fuzzy = pd.read_parquet("rc_ac_matches.parquet")
cluster = pd.read_parquet("rc_ac_cluster_matches_20260517.parquet")
clusters_csv = pd.read_csv(
    "cluster_assignments_20260517.csv",
    usecols=["company_name", "global_cluster_id", "cluster_size"],
    low_memory=False,
)

print(f"fuzzy pairs:   {len(fuzzy):,}")
print(f"cluster pairs: {len(cluster):,}")


fuzzy pairs:   50,472
cluster pairs: 60,731


## Identify the three source cells

In [2]:
# Use a string pair key for fast set operations
fuzzy["_pkey"] = fuzzy.r_case_number + "|" + fuzzy.c_case_number
cluster["_pkey"] = cluster.r_case_number + "|" + cluster.c_case_number

fuzzy_pkeys = set(fuzzy._pkey)
cluster_pkeys = set(cluster._pkey)

agreement = fuzzy_pkeys & cluster_pkeys
fuzzy_only = fuzzy_pkeys - cluster_pkeys
cluster_only = cluster_pkeys - fuzzy_pkeys

print(f"agreement:    {len(agreement):,}")
print(f"fuzzy_only:   {len(fuzzy_only):,}")
print(f"cluster_only: {len(cluster_only):,}")


agreement:    49,426
fuzzy_only:   1,046
cluster_only: 11,305


## Build cluster_size lookup

The cluster representative (used in matching) is the shortest name per cluster
(same logic as `add_cluster_representatives.py`). We map each representative
to its cluster size so we can stratify the cluster-only sample.


In [3]:
reps = (
    clusters_csv.sort_values("company_name", key=lambda s: s.str.len())
    .drop_duplicates(subset="global_cluster_id", keep="first")
)
rep_to_size = dict(zip(reps.company_name, reps.cluster_size))
print(f"Representatives indexed: {len(rep_to_size):,}")
print(f"Sample sizes available: min={reps.cluster_size.min()}, "
      f"max={reps.cluster_size.max()}, median={int(reps.cluster_size.median())}")


Representatives indexed: 91,120
Sample sizes available: min=1, max=187, median=2


## Sample each stratum (seeded)

In [4]:
rng_state = SEED  # used as random_state base; sub-strata use SEED+k

# --- Agreement: 50, restricted to non-trivial pairs (preprocessed names differ) ---
# Drawn from fuzzy rows; match_company_r/c in fuzzy parquet ARE the preprocessed names.
agreement_pool = fuzzy[
    fuzzy._pkey.isin(agreement)
    & (fuzzy.match_company_r != fuzzy.match_company_c)
].copy()
agreement_sample = agreement_pool.sample(n=50, random_state=rng_state)
print(f"agreement pool (non-trivial): {len(agreement_pool):,}  -> sampled 50")

# --- Fuzzy-only: 100, stratified by fuzzy_score band ---
fuzzy_only_pool = fuzzy[fuzzy._pkey.isin(fuzzy_only)].copy()
low_band  = fuzzy_only_pool[(fuzzy_only_pool.fuzzy_score >= 82) & (fuzzy_only_pool.fuzzy_score < 90)]
high_band = fuzzy_only_pool[fuzzy_only_pool.fuzzy_score >= 90]
print(f"fuzzy_only pool: {len(fuzzy_only_pool):,}  "
      f"(low [82,90): {len(low_band):,}, high [90,100]: {len(high_band):,})")

fuzzy_low_sample  = low_band.sample(n=50, random_state=rng_state + 1)
fuzzy_high_sample = high_band.sample(n=50, random_state=rng_state + 2)

# --- Cluster-only: 100, stratified by cluster_size ---
cluster_only_pool = cluster[cluster._pkey.isin(cluster_only)].copy()
# Cluster representative is in match_company_r (== match_company_c) for cluster output
cluster_only_pool["cluster_size"] = (
    cluster_only_pool["match_company_r"].map(rep_to_size).fillna(1).astype(int)
)
small = cluster_only_pool[cluster_only_pool.cluster_size <= 5]
large = cluster_only_pool[cluster_only_pool.cluster_size > 5]
print(f"cluster_only pool: {len(cluster_only_pool):,}  "
      f"(size<=5: {len(small):,}, size>5: {len(large):,})")

cluster_small_sample = small.sample(n=50, random_state=rng_state + 3)
cluster_large_sample = large.sample(n=50, random_state=rng_state + 4)


agreement pool (non-trivial): 9,688  -> sampled 50
fuzzy_only pool: 1,046  (low [82,90): 681, high [90,100]: 365)
cluster_only pool: 11,305  (size<=5: 5,581, size>5: 5,724)


## Assemble visible CSV + hidden key

In [5]:
# Annotate each sample with its source_cell and stratum
agreement_sample = agreement_sample.copy()
agreement_sample["source_cell"] = "agreement"
agreement_sample["stratum"] = "agreement"
agreement_sample["cluster_size"] = np.nan

fuzzy_low_sample = fuzzy_low_sample.copy()
fuzzy_low_sample["source_cell"] = "fuzzy_only"
fuzzy_low_sample["stratum"] = "score_82_90"
fuzzy_low_sample["cluster_size"] = np.nan

fuzzy_high_sample = fuzzy_high_sample.copy()
fuzzy_high_sample["source_cell"] = "fuzzy_only"
fuzzy_high_sample["stratum"] = "score_90_100"
fuzzy_high_sample["cluster_size"] = np.nan

cluster_small_sample = cluster_small_sample.copy()
cluster_small_sample["source_cell"] = "cluster_only"
cluster_small_sample["stratum"] = "cluster_size_le5"

cluster_large_sample = cluster_large_sample.copy()
cluster_large_sample["source_cell"] = "cluster_only"
cluster_large_sample["stratum"] = "cluster_size_gt5"

all_samples = pd.concat(
    [agreement_sample, fuzzy_low_sample, fuzzy_high_sample,
     cluster_small_sample, cluster_large_sample],
    ignore_index=True,
)

# Shuffle to fully randomize order (so adjacency doesn't reveal source_cell)
all_samples = all_samples.sample(frac=1, random_state=SEED).reset_index(drop=True)
all_samples.insert(0, "pair_id", range(1, len(all_samples) + 1))

print(f"Total sampled: {len(all_samples)}")
print()
print("Breakdown:")
print(all_samples.groupby(["source_cell", "stratum"]).size().to_string())


Total sampled: 250

Breakdown:
source_cell   stratum         
agreement     agreement           50
cluster_only  cluster_size_gt5    50
              cluster_size_le5    50
fuzzy_only    score_82_90         50
              score_90_100        50


## Save files

In [6]:
VISIBLE_COLS = [
    "pair_id",
    "r_case_number", "c_case_number",
    "r_company_name", "c_company_name",
    "r_state", "r_city",
    "c_state", "c_city",
    "r_date_filed", "r_date_closed", "c_date_filed",
]

visible = all_samples[VISIBLE_COLS].copy()
visible["label"] = ""    # to fill: match | no_match | unclear
visible["notes"] = ""    # optional: free-text reasoning

key = all_samples[
    ["pair_id", "source_cell", "stratum",
     "cluster_size", "fuzzy_score", "match_method"]
].copy()

visible.to_csv("evaluation_sample.csv", index=False)
key.to_csv("evaluation_sample_key.csv", index=False)

print("Saved evaluation_sample.csv      (250 rows, columns hidden from method info)")
print("Saved evaluation_sample_key.csv  (250 rows, unblinding key)")


Saved evaluation_sample.csv      (250 rows, columns hidden from method info)
Saved evaluation_sample_key.csv  (250 rows, unblinding key)


## Labeling instructions

1. Open `evaluation_sample.csv` (in Excel, Google Sheets, or a text editor).
2. For each row, fill in the `label` column with one of:
   - **`match`**     — the R-case and C-case clearly refer to the **same workplace**
   - **`no_match`**  — the R-case and C-case clearly refer to **different workplaces**
   - **`unclear`**   — not enough information to decide (use sparingly)
3. Optionally add reasoning to the `notes` column, especially for `no_match` or `unclear`.
4. Save the file back as CSV.

**Do NOT open `evaluation_sample_key.csv` until you are done labeling.**
It reveals which method flagged each pair, and reading it first would bias your judgment.

When done, run `analyze_evaluation_results.ipynb` to unblind and compute precision.

### Tips for labeling
- Same company name + same city + overlapping dates → almost always `match`.
- Different industries/business types at different addresses → `no_match`.
- Parent/subsidiary, DBA relationships, division names → typically `match` (same workplace).
- Two completely different companies that happen to have similar names → `no_match`.
- If addresses are missing or names are too generic ("ABC Inc.") → `unclear`.
